### RAG Pipelines - Data Ingestion to Vector DB PipelineCreation 

In [ ]:
### Just for checking the version of Python and Langchain
import sys
print(sys.executable)
import langchain
print(langchain.__version__)

c:\rag-project\venv\Scripts\python.exe
1.2.15


Actual change starts here

In [ ]:
### Importing necessary libraries

import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

Data Ingestion: - Reading all the pdf documents

In [ ]:
### Read all the pdf's inside the pdf directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: sample.pdf
  ✓ Loaded 417 pages

Processing: sample1.pdf
  ✓ Loaded 1 pages

Processing: sample2.pdf
  ✓ Loaded 1 pages

Processing: sample3.pdf
  ✓ Loaded 1 pages

Total documents loaded: 420


In [21]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat PDFMaker 15 for PowerPoint', 'creator': 'PyPDF', 'creationdate': '2026-02-11T09:06:14+00:00', 'moddate': '2026-02-11T14:36:58+05:30', 'source': '..\\data\\pdf_files\\sample.pdf', 'total_pages': 417, 'page': 0, 'page_label': '1', 'source_file': 'sample.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Acrobat PDFMaker 15 for PowerPoint', 'creator': 'PyPDF', 'creationdate': '2026-02-11T09:06:14+00:00', 'moddate': '2026-02-11T14:36:58+05:30', 'source': '..\\data\\pdf_files\\sample.pdf', 'total_pages': 417, 'page': 1, 'page_label': '2', 'source_file': 'sample.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'Acrobat PDFMaker 15 for PowerPoint', 'creator': 'PyPDF', 'creationdate': '2026-02-11T09:06:14+00:00', 'moddate': '2026-02-11T14:36:58+05:30', 'source': '..\\data\\pdf_files\\sample.pdf', 'total_pages': 417, 'page': 2, 'page_label': '3', 'source_file': 'sample.pdf', 'file_type': 'pdf'}, page

Chunking

In [24]:
### Text splitting get into chunks
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, ## number of characters to include in each chunk
        chunk_overlap=chunk_overlap, ## number of characters to overlap between chunks
        length_function=len, ## use the default length function which counts characters
        separators=["\n\n", "\n", " ", ""] ## separators to split the text, in order of priority
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"Example chunk:")
        print(f"Example chunk: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [25]:
chunks = split_documents(all_pdf_documents)
chunks

Split 420 documents into 7 chunks
Example chunk:
Example chunk: Lav Kush
Bengaluru, Karnataka
 
lavkush.pnbe@gmail.com
 
7250 338280
 
https://www.linkedin.com/in/lav-kush-982546190
 
Professional Experience
Tata Consultancy Services (TCS) - (Backend Developer) Au...
Metadata: {'producer': 'Skia/PDF m135', 'creator': 'FlowCV - https://flowcv.com', 'creationdate': '2025-09-21T12:14:32+00:00', 'moddate': '2025-09-21T12:14:32+00:00', 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'source': '..\\data\\pdf_files\\sample1.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m135', 'creator': 'FlowCV - https://flowcv.com', 'creationdate': '2025-09-21T12:14:32+00:00', 'moddate': '2025-09-21T12:14:32+00:00', 'keywords': 'FlowCV – Online Resume Builder – https://flowcv.com', 'source': '..\\data\\pdf_files\\sample1.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sample1.pdf', 'file_type': 'pdf'}, page_content='Lav Kush\nBengaluru, Karnataka\n \nlavkush.pnbe@gmail.com\n \n7250 338280\n \nhttps://www.linkedin.com/in/lav-kush-982546190\n \nProfessional Experience\nTata Consultancy Services (TCS) - (Backend Developer) Aug\xa02023 – Present | Bengaluru\nInsurance Middleware System Development and Integration | JAVA, Spring Boot, JPA, IBM IID |\n Client: HDFC Life \n• Designed and implemented 10+ microservices and RESTful APIs using Java, Spring Boot to manage policy data, claims \nprocessing, and customer information within the middleware layer. \n• Collaborated with a team of 10 developers and cross-f

Embedding using open source model

In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\rag-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__()